# ion_gym — general simulation workbench

One interactive app for **any** geometry, driven by `build_run(spec)`:
the planar (Cartesian) einzel, the round (r-z) einzel, the IMS drift
tube, and the funnel all load into the same interface. Nothing external needed
to run (only the shipped funnel uses pre-solved bases).

Features: Start/Stop at the top; controls organized in **tabs**
(Source / Voltages / Gas / Integration / Display / Config); **free-aspect
zoom** (box-zoom any size and shape, scroll to zoom, double-click to
reset); equipotential + optional |E| field preview *before* flying;
per-electrode voltage adjustment (re-weight, no re-solve); and **stored,
reloadable runs** — a completed simulation is kept and can be reloaded,
re-styled, re-colored, and exported (not one-and-done).

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# ---- named parameters (units and rationale; nothing below is a magic
# literal) ------------------------------------------------------------
DECK_NAME = 'einzel_round_r-z.json'   # any deck in examples/; the menu
                                      # and this notebook read the SAME
                                      # JSON through the SAME loader
LENS_V = -200.0                       # centre-electrode bias [V]. A
                                      # VOLTAGE change reweights cached
                                      # bases (>200x faster than a solve),
                                      # so retuning never rebuilds geometry
LENS_ELECTRODE = 'centre'             # name as declared in the deck

# Examples are JSON decks: load the shipped deck, report what
# it carries vs what this notebook overrides, then hand it to the app.
from pathlib import Path
from ion_gym.io.paths import repo_root
from ion_gym.io.sim_spec import SimSpec
from ion_gym.io.deck_params import apply_deck_overrides
from ion_gym.ui.sim_app import SimApp
spec2 = SimSpec.from_json(str(Path(repo_root()) / 'examples' / DECK_NAME))
apply_deck_overrides(spec2, dc={LENS_ELECTRODE: LENS_V})
app2 = SimApp(spec2)
# app2.panel().show()
print('loaded:', app2.spec.name, '| coords:', app2.spec.geometry.coords)

## Python dependencies
Core: `numpy`, `scipy`, `numba` &nbsp;·&nbsp; App/plots: `panel`, `plotly`, `bokeh`, `pandas` &nbsp;·&nbsp; STL path: `trimesh`, `manifold3d`

Run the cell below once per environment (matches `requirements.txt`).


In [ ]:
# %pip install -q numpy scipy numba panel plotly bokeh pandas trimesh manifold3d
import importlib, sys
req = {'numpy':None,'scipy':None,'numba':None,'panel':None,'plotly':None,
       'bokeh':None,'pandas':None,'trimesh':'STL path','manifold3d':'STL path'}
missing = []
for m, note in req.items():
    try:
        v = getattr(importlib.import_module(m), '__version__', 'installed')
        print(f'{m:<12} {v}')
    except Exception:
        missing.append(m)
        print(f'{m:<12} MISSING' + (f'  ({note})' if note else ''))
if missing:
    print('\ninstall with:  %pip install ' + ' '.join(missing))


## Data location

In [ ]:
# repo root on path from ANYWHERE (the notebook
# moved to notebooks/, so '.' is no longer the root):
import sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'pyproject.toml').exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))
# resolve WITHOUT `pip install -e .`, as long as the notebook runs from the repo
# root. Installing the package makes it work from any directory.
from ion_gym.io import paths
# paths.set_data_root('/path/to/data')  # only needed for the shipped funnel
print('data root:', paths.data_root())

## Launch

`serve_dashboard` puts all three pages on ONE server: the app at `/`, the
geometry editor at `/editor`, and the 3-D flight viewer at `/flight`. Use
it rather than `app.panel().show()` — that starts a server with only `/`,
so the app's own **Edit geometry** and **Flight view** buttons, which
link to those paths, answer **404**.

**VS Code:** run the cell; it opens a browser tab.
**JupyterLab/classic:** the same call works, or return `app.panel()` to
display inline — inline gives you the app alone, without the two
buttons' pages.

In [ ]:
from ion_gym.ui.sim_app import SimApp
from ion_gym.ui.serve import serve_dashboard

app = SimApp()          # starts on the funnel example
# ONE server, three routes: / (app), /editor, /flight -- so the app's
# "Edit geometry" and "Flight view" buttons resolve instead of 404ing.
# threaded=True returns immediately, so the cells below this one still
# run; the server keeps going until the kernel stops.
server = serve_dashboard(app=app, port=5006, show=True, threaded=True)
# JupyterLab/classic inline alternative (app only, no /editor or
# /flight): app.panel()
# Stop it with: server.stop()

## Loading a geometry

Use the **Config** tab's *load example* to switch between the planar
einzel, round einzel, IMS, and funnel — or paste/upload a spec JSON. From
code:

In [ ]:
# ---- named parameters (units and rationale; nothing below is a magic
# literal) ------------------------------------------------------------
DECK_NAME = 'einzel_round_r-z.json'   # any deck in examples/; the menu
                                      # and this notebook read the SAME
                                      # JSON through the SAME loader
LENS_V = -200.0                       # centre-electrode bias [V]. A
                                      # VOLTAGE change reweights cached
                                      # bases (>200x faster than a solve),
                                      # so retuning never rebuilds geometry
LENS_ELECTRODE = 'centre'             # name as declared in the deck

# Examples are JSON decks: load the shipped deck, report what
# it carries vs what this notebook overrides, then hand it to the app.
from pathlib import Path
from ion_gym.io.paths import repo_root
from ion_gym.io.sim_spec import SimSpec
from ion_gym.io.deck_params import apply_deck_overrides
from ion_gym.ui.sim_app import SimApp
spec2 = SimSpec.from_json(str(Path(repo_root()) / 'examples' / DECK_NAME))
apply_deck_overrides(spec2, dc={LENS_ELECTRODE: LENS_V})
app2 = SimApp(spec2)
# app2.panel().show()
print('loaded:', app2.spec.name, '| coords:', app2.spec.geometry.coords)

## Free-aspect zoom

The plot uses `dragmode='zoom'` with independent autorange on both axes,
so a box-zoom can be **any size and any aspect ratio** — drag a tall thin
box or a wide short one. Scroll to zoom, double-click to reset. Tick
*lock aspect ratio* on the Display tab only if you want physical 1:1.

## Reloading trajectories after a run

Every completed run is stored (timestamped) and listed in the **Config**
tab under *stored runs*. Select one and **Reload run into view** to bring
it back — then change the color-by channel, trajectory style, or splat
markers on the Display tab and it re-renders from the stored data.
**Export displayed run** writes all recorded channels to CSV. From
code:

## Clear / Recompute / Reset

The top row has **Clear ions** (drop displayed trajectories, keep stored
runs and the field), **Recompute field** (re-solve/re-weight from the
current controls — picks up a geometry or voltage change), and **Reset
app** (back to the spec the app was created with, dropping stored runs).
Loading a new example or JSON now correctly rebuilds the per-electrode
Voltages tab (an einzel shows entrance/lens/exit, not the funnel rings).

## Bounding / splat planes

The **Bounds** tab adds optional x/y/z min/max planes — an ion crossing an
*enabled* plane terminates (fate: *bounding plane*, purple). All off by
default. This solves the "ions fly off to infinity" problem (e.g. an
einzel with no downstream wall): set an `x max` plane at the detector
distance and ions splat there instead of coasting out. In r-z, x is the
axis (so `x max` is a detector-plane distance) and y is the radius (so
`y max` is a radial aperture). Enabled planes draw as dashed purple
guides.

In [ ]:
# after running in the UI:
# df = app.results_dataframe()          # the displayed run's channels
# df = app.results_dataframe('einzel — round (r-z) [14:22:07]')  # a named run
# df.to_parquet('run.parquet')
print('stored runs live in app._runs; results_dataframe() exports any of them')

## RF phase groups

DC and RF are independent per electrode: each has a **DC value** (always
applied) and an optional **RF-group** membership. So DC-only, RF-only, and
RF+DC are all just combinations of those two. Define groups once (amplitude
/ frequency / phase) in the **Voltages** tab, then assign each electrode to
a group or *(none)* via its dropdown. Two groups 180° apart is a funnel or
a quad rod-pair; many groups at stepped phases is a **sinusoidal
travelling wave** (pigsim-style) — build them with
`travelling_wave_groups(n)` and assign electrodes cyclically. From code:

In [ ]:
from ion_gym.io.sim_spec import RFGroupSpec, travelling_wave_groups
# two-group (funnel/quad):
groups = [RFGroupSpec('RFA', frequency_hz=5e5, amplitude_v=50, phase_deg=0),
          RFGroupSpec('RFB', frequency_hz=5e5, amplitude_v=50, phase_deg=180)]
# electrode.rf_group = 'RFA' or 'RFB'; electrode.dc set independently

# sinusoidal travelling wave (8 groups at 45 deg steps):
tw = travelling_wave_groups(8, frequency_hz=1e6, amplitude_v=30)
print([(g.name, g.phase_deg) for g in tw])
# assign electrode k -> tw[k % 8] to march the wave

## View planes (xy / xz / yz)

The **Display** tab has a *view plane* selector. Because every trajectory
stores x, y **and** z per step, switching planes re-projects the **same
stored run instantly** — no re-flying. The solved field background is shown
in the xy plane (where the 2-D solve defines it); xz/yz show the
trajectories alone (no fabricated field). For the r-z funnel this lets you
see the beam's transverse structure; for the eventual 3-D quad all three
planes carry distinct information.

## Multiple ion masses (per-ion m/z)

`source.mz_list` can hold several masses; ions cycle through them, and
each now flies with **its own mass** (the planar tracer computes per-ion
acceleration). So one run can mix species — e.g. a stable and an unstable
quadrupole ion at the same RF. In the app, the Source tab's m/z field
sets a single mass; for multiple, edit `mz_list` in the Config-tab JSON.

## STL electrodes

An electrode can take its geometry from an **STL file** instead of inline
shapes (set `electrode.stl` and `geometry.stl_dir`). The STL is voxelized
(Rung-2 voxelizer) and, for a 2-D
study, sliced at the mid-plane to a cross-section mask — then solved by
the same native solver. Validated on the einzel: STL plates reproduce the
inline field to 0.6 V and the same focusing. Build via
`build_stl.build_stl_run(spec)`; for open geometries (no enclosing wall)
pass `ground_border=True`.

## Quadrupole (STL rods): stable vs unstable

The example **"quadrupole — STL rods (stable+unstable)"** builds four
cylindrical rods from STL, forms the RF quadrupole (x-rods and y-rods in
antiphase — one signed RF basis), and flies two masses at once: m/z 100
(Mathieu q~0.40, **stable** — stays bounded) and m/z 30 (q~1.32,
**unstable** — ejected in <1 us). Bounds at the rod radius make the
unstable ion splat (fate 3). This exercises the STL path, the signed
two-phase fold, per-ion mass, and RF flight together.

Note: the planar field solve runs on a 3-cell z-slab (not a single
plane). A single-plane solve has an x/y asymmetry in the iterative solver;
the 3-slab solve matches the direct 2-D solver to ~2% (planar einzel
-73V vs -71.4V ground truth).

## Quadrupole as axial transport

The quad example is now an **axial-transport** quad:
ions are injected near-axis with axial KE and drift down the transport
axis (z) while the RF confines them radially. Because an ideal quad's
axial motion decouples (no z-force), the drift `z = z0 + vz*t` is exact on
top of the validated 2-D transverse solve — no 3-D solve needed.

At 150 V / 2 MHz, m/z 100 (Mathieu q~0.40) **transmits** — it wiggles and
drifts the full length while staying within ~0.7 mm of axis — while m/z 30
(q~1.32) is **lost**, its radial excursion growing until it strikes a rod
near the entrance. View in the **xz** plane to see the transport wiggle;
the x-rods appear as the channel walls. (The planar tracer carries an
optional field-free axial drift; set it via a source `direction` with a
z-component and axial KE. An einzel has no z-drift and is unaffected.)

Aspect: the cross-section (xy) view is auto-locked so round rods render
round; the legend sits horizontally above the plot so it never squishes
the aspect.

## Performance: the STL build is cached

The STL path solves per-electrode field bases by SOR, which is a few
seconds on a fine grid. That solve is now **cached on the geometry**: the
first build for a geometry solves the bases, and every later build with
the same geometry but different **voltages or RF** re-weights the cached
bases in milliseconds (the fast-adjust invariant). Changing the geometry
(grid resolution, STL files) re-solves.

So in the app: the first Fly on a new geometry takes a few seconds to
solve (shown in the status bar); after that, voltage tweaks, view
switches, and re-flies are instant. Tick **verbose build log** to print
stage timings to the console — e.g. `[stl] solved 5 bases on 145x145:
voxelize 0.03s + solve 3.7s (cached for reuse)` then `[stl] CACHE HIT`.

The quad demo grid is 0.15 mm (≈4 s solve); 0.1 mm was ~5x slower for a
<1% field change, not worth it for a demo. If you need the finer field for
a production run, set `geometry.mm_per_gu = 0.1`.

## Solving spinner

When a geometry needs a fresh field solve (a few seconds the first time),
the app now runs the solve on a **background thread** and shows a spinner
over the plot plus a *"solving field…"* status — so it's clear the app is
working, not hung. The solver releases the GIL, so the UI stays live and
the spinner animates. Once solved, the bases are cached: voltage tweaks,
view switches, and re-flies skip the solve entirely and build inline in
milliseconds (no spinner flicker — the app only shows it when the build
will actually be slow).

## Quad geometry & fields (r0, resolution, viewing)

The quad example geometry: **r0 = 3.84 mm** inscribed
radius, round rods with R = 1.148·r0 (the classic ratio), following
ion_playground's convention. The RF amplitude is set **analytically** from
the Mathieu relation V = q·m·r0²·Ω²/(4e) for a target q on the stable mass
(no empirical field fit), so at q = 0.40 for m/z 100 (V ≈ 241 V @ 2 MHz),
m/z 30 sits at q ≈ 1.33 and is lost.

**Fields:** a quad is pure-RF (all rods DC = 0), so the *static* field is
zero — the saddle lives in the RF component. The field display now shows
the static + RF-peak field, so switching to the **xy** view with *show
field* on reveals the quadrupole saddle (|E| = 0 on axis, rising outward).
The field is transverse, so it only renders in xy; a hint appears if you
ask for it from a transport view.

**STL resolution:** the voxelizer rasterizes the mesh onto the grid, so the
binding resolution is `mm_per_gu` (the grid), not the STL facet count — a
rod's area is captured accurately even at low facet counts. The chunky look
at 0.15 mm is grid stair-stepping; drop to a finer grid for smoother
electrodes (cheap to iterate now that the solve is cached). High-facet STLs
voxelize fine (~tens of ms, one-time).

## Editing the m/z list

`source.mz_list` holds one or more masses; ions cycle through them, each
flown with its **own** mass (e.g. a stable + unstable quad ion in one run).
Two ways to set it:
- **GUI**: the Source tab has an *"m/z list (Da, comma-separated)"* field —
  type `100, 200, 500` (spaces/semicolons/trailing commas are tolerated).
- **JSON** (Config tab): edit `"mz_list": [100, 200, 500]` in the source
  block and click apply.

Because the tracer computes each ion's mass and acceleration per index,
one ensemble can mix species — no re-solve needed (the field is cached;
only the per-ion flight differs).

## PE surface (effective potential)

The Display tab's **field shading** selector now offers **"PE surface
(effective)"**: the potential-energy landscape the ion moves on, in eV —
hills repel, valleys confine.

- **DC devices** (einzel, reflectron): PE = charge x the static potential —
  the ion literally rolls on it.
- **RF devices** (quad, funnel): the instantaneous potential flips every
  half-cycle, so the view shows the time-averaged **Dehmelt
  pseudopotential** V = q|E0|²/(4mΩ²) — always a confining well, zero at
  the RF null, **mass-dependent** (heavier ions see a shallower well — the
  mass filter). Set the ion mass with the **PE m/z** input.
- **Combined**: DC + pseudo, e.g. the funnel's DC push riding inside its
  RF wall.

Validated: the quad's well depth matches the classic identity
D = q_Mathieu·V_rf/4 to 4%, and the mass scaling is exactly 1/m. The view
is labelled *effective (adiabatic) potential* — it's an approximation
valid for Mathieu q ≲ 0.4; near the stability boundary the full RF tracer
is ground truth. Computed from the cached bases, so switching mass is
instant.

## Read-out

- **The workbench is the same code path as every notebook**, driven interactively: `build_run(spec)` solves, flies, and renders exactly as a scripted run does. That equivalence is the point — if a device behaves differently here than in a notebook, the difference is a bug, not a feature of the GUI.
- **Use it for exploration, not for results.** An interactively tuned operating point is a hypothesis; it becomes a result when it is written into a deck and re-flown from that deck with its parameters recorded. The export button exists to make that step one click.
- **This notebook drives the toolkit rather than analysing one instrument**, so it carries no fixed geometry panel by design (declared as tooling in the notebook audit); the device it shows is whatever `DECK` points at.